In [ ]:
import  numpy as np

sls = np.array([
    [12000,  8500, 15000, 22000,  6000],   # Q1
    [14000,  9200, 13500, 24000,  7200],   # Q2
    [11500,  8000, 16000, 20000,  8500],   # Q3
    [16000, 10500, 14500, 26000,  9000]    # Q4
])
print(sls.shape)


(4, 5)
[53500 36200 59000 92000 30700]


In [4]:
prodTl = sls.sum(axis=0)
print(prodTl)
print(prodTl.shape)

[53500 36200 59000 92000 30700]
(5,)


In [3]:
qytrTls = sls.sum(axis=1)
print(qytrTls)
print(qytrTls.shape)

[63500 67900 64000 76000]
(4,)


In [6]:
sales = np.array([
    [12000,  8500, 15000, 22000,  6000],
    [14000,  9200, 13500, 24000,  7200],
    [11500,  8000, 16000, 20000,  8500],
    [16000, 10500, 14500, 26000,  9000]
])

# Which product had the highest single-quarter sales?
print(sales.max(axis=0))

# Which quarter had the lowest single-product sales?
print(sales.min(axis=1))

# Average quarterly revenue per product
print(sales.mean(axis=0).astype(int))

# Which quarter had the highest total revenue?
best_quarter = sales.sum(axis=1).argmax()
print(f"Best quarter: Q{best_quarter + 1}")

print(sales.std(axis=0).astype(int))


[16000 10500 16000 26000  9000]
[6000 7200 8000 9000]
[13375  9050 14750 23000  7675]
Best quarter: Q4
[1780  939  901 2236 1169]


#### keepdims — Preserving Shape After Aggregation
By default, when you aggregate along an axis, that dimension disappears from the result. Sometimes you need the result to keep the original number of dimensions, but with size 1 where the collapse happened. That is what keepdims=True does.

In [14]:
dta = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print(dta.shape)
row = dta.sum(axis=1)
print(row)
print(row.shape)

rkept = dta.sum(axis=1, keepdims=True)
print(rkept)
print(rkept.shape)

(3, 3)
[ 60 150 240]
(3,)
[[ 60]
 [150]
 [240]]
(3, 1)


In [ ]:
import numpy as np

arr = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3])

# Cumulative sum — each element is the running total
print(np.cumsum(arr))
# Output: [ 3  4  8  9 14 23 25 31 36 39]

# Index of the minimum and maximum values
print(np.argmin(arr))   # 1 — position of the smallest value (1)
print(np.argmax(arr))   # 5 — position of the largest value (9)

# Percentile — what value is at the 75th percentile?
print(np.percentile(arr, 75))   # 5.25

# Median
print(np.median(arr))   # 3.5

#### Broadcasting
You have a 2D array and you want to do arithmetic between it and a 1D array. Mathematically you know what you mean — maybe you want to subtract the column mean from every row, or normalize each column by its maximum. But the shapes do not match.

In [15]:
import numpy as np

data = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])
# shape: (3, 3)

column_means = data.mean(axis=0)
print(column_means)   # [40. 50. 60.]
# shape: (3,)

# This works — even though shapes are different
centered = data - column_means
print(centered)
# Output:
# [[-30. -30. -30.]
#  [  0.   0.   0.]
#  [ 30.  30.  30.]]

[40. 50. 60.]
[[-30. -30. -30.]
 [  0.   0.   0.]
 [ 30.  30.  30.]]


In [2]:
import numpy as np

dta = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

rws = np.array([100, 200, 300])
print(dta.shape)
print(rws.shape)

res = dta + rws
print(res)
print(res.shape)

(3, 3)
(3,)
[[101 202 303]
 [104 205 306]
 [107 208 309]]
(3, 3)


where shape matters critically, and where keepdims=True becomes essential:


In [4]:
sls = np.array([
    [12000,  8500, 15000],
    [14000,  9200, 13500],
    [11500,  8000, 16000]
])

cols = sls.max(axis=0)
print(cols)

normcols = sls / cols
print(normcols.round(2))

[14000  9200 16000]
[[0.86 0.92 0.94]
 [1.   1.   0.84]
 [0.82 0.87 1.  ]]


That worked because (3,) pads to (1, 3) and stretches to (3, 3) — matching the column dimension.
Now try normalising each ROW by its maximum:

In [6]:
import numpy as np

sales = np.array([
    [12000,  8500, 15000],
    [14000,  9200, 13500],
    [11500,  8000, 16000]
])

row_maxes = sales.max(axis=1)           # shape: (3,)
print(row_maxes)                         # [15000 14000 16000]

# This will FAIL — let us trace why
# sales shape:     (3, 3)
# row_maxes shape: (3,)
# Pad: (3,) becomes (1, 3)
# Try to stretch: (1, 3) against (3, 3)
#   - first dim: 1 stretches to 3 — fine
#   - second dim: 3 vs 3 — fine, no stretching needed
# BUT this means we are dividing by (3,) treated as columns, not rows!
# The result is mathematically wrong, not an error.
print(sales / row_maxes)
# Output: wrong normalisation — each element is divided by the wrong row's max

[15000 14000 16000]
[[0.8        0.60714286 0.9375    ]
 [0.93333333 0.65714286 0.84375   ]
 [0.76666667 0.57142857 1.        ]]


That silent error is dangerous. NumPy did not crash. It just gave the wrong answer because the shapes lined up accidentally. This is exactly the class of bug that causes incorrect results in real data pipelines and is hard to catch because there is no error message.
The fix is keepdims=True:


In [5]:
import numpy as np

sales = np.array([
    [12000,  8500, 15000],
    [14000,  9200, 13500],
    [11500,  8000, 16000]
])

row_maxes = sales.max(axis=1, keepdims=True)   # shape: (3, 1) not (3,)
print(row_maxes)
# Output:
# [[15000]
#  [14000]
#  [16000]]

# Now broadcasting works correctly
# (3, 3) / (3, 1)
# (3, 1) stretches its column dimension to 3 → (3, 3)
# Each row is divided by its own maximum
normalised_rows = sales / row_maxes
print(normalised_rows.round(2))
# Output:
# [[0.8  0.57 1.  ]
#  [1.   0.66 0.96]
#  [0.72 0.5  1.  ]]

[[15000]
 [14000]
 [16000]]
[[0.8  0.57 1.  ]
 [1.   0.66 0.96]
 [0.72 0.5  1.  ]]


In [7]:
data = np.array([
    [2.5, 1000, 0.3, 85],
    [3.1, 1200, 0.7, 90],
    [1.8,  950, 0.2, 78],
    [4.2, 1500, 0.9, 92],
    [2.9, 1100, 0.5, 88]
], dtype=float)

colMns = data.mean(axis=0)
colStd = data.std(axis=0)

stzd = (data - colMns) / colStd
print(stzd.round(2))
print("\nColumn means after standardising:", stzd.mean(axis=0).round(10))
print("Column stds after standardising:", stzd.std(axis=0).round(2))

[[-0.51 -0.77 -0.86 -0.33]
 [ 0.25  0.26  0.7   0.7 ]
 [-1.4  -1.03 -1.25 -1.76]
 [ 1.65  1.8   1.48  1.11]
 [ 0.   -0.26 -0.08  0.29]]

Column means after standardising: [ 0.  0. -0.  0.]
Column stds after standardising: [1. 1. 1. 1.]
